### Preliminary cell to start the notebook

In [ ]:
# libraries
import os
import sys

print(sys.version)

in_colab = "google.colab" in sys.modules
if in_colab:
    if not os.getcwd().split("/")[-1].split("_")[-1] == "2023":
        from google.colab import drive

        drive.mount("/content/drive")
        os.chdir(r"/content/drive/MyDrive/Human_Data_Analytics_Project_2023")

    if not "tensorflow_io" in sys.modules:
        print("Installing tensorflow-IO")
        !pip install tensorflow-io
    if not "keras" in sys.modules:
        print("Installing keras")
        !pip install keras==2.10.0
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0

main_dir = os.getcwd()
if main_dir not in sys.path:
    print("Adding the folder for the modules")
    sys.path.append(main_dir)

# BASE LIBRARIES
import numpy as np
import pandas as pd
import h5py
import shutil
import time
import random
import subprocess
import itertools
import warnings
import pickle
import json

# PLOT LIBRARIES
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
import IPython.display as ipd

# import plotly.express as px

# AUDIO LIBRARIES
import librosa
from scipy.io import wavfile
from scipy import signal
from scipy.fft import fft, ifft, fftfreq, fftshift
from scipy.signal import stft, spectrogram, periodogram

# from pydub import AudioSegment

# MACHINE LEARNING LIBRARIES
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, LeaveOneOut, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.utils import check_random_state
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from keras import layers
from keras import models
from keras.utils import plot_model as tf_plot

if in_colab:
    import tensorflow_io as tfio
print("TensorFlow version:", tf.__version__)
# show keras version
import keras

print(f"keras version = {keras.__version__}")
# import keras_tune as kt
from keras import layers
import keras_tuner as kt
from tensorflow import keras
from keras.regularizers import L1L2

# kernel_regularizer=regularizers.L1L2(l1=1e-5, l2=1e-4) # we may use this in some layers...

# RANDOM SETTINGS
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
check_random_state(seed)

# EVALUATION LIBRAIRES
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_curve
from sklearn.metrics import make_scorer
from sklearn.metrics import (
    RocCurveDisplay,
    precision_recall_curve,
    PrecisionRecallDisplay,
)
from sklearn.metrics import precision_recall_fscore_support, auc

# OUR PERSONAL FUNCTIONS
import importlib
from Preprocessing.data_loader import download_dataset, load_metadata
from Preprocessing.exploration_plots import (
    one_random_audio,
    plot_clip_overview,
    Spectral_Analysis,
)
from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Visualization.model_plot import confusion_matrix, listen_to_wrong_audio

importlib.reload(importlib.import_module("Preprocessing.data_loader"))
importlib.reload(importlib.import_module("Models.basic_ml"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Preprocessing.data_loader import load_metadata

df_ESC10, df_ESC50 = load_metadata(
    main_dir, heads=False, ESC_US=False, statistics=False
)

from Preprocessing.data_loader import load_metadata
from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)

importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Models.ann_utils import *
from Models.ann_utils import MFCCWithDeltaLayer, OutputCutterLayer
from Visualization.model_plot import (
    plot_history,
    confusion_matrix,
    listen_to_wrong_audio,
    visualize_the_weights,
)

ESC10_path = os.path.join(main_dir, "data", "ESC-10-depth")
samplerate = 44100

## 2.4 Recurrent Neural Networks

In [ ]:
import importlib

importlib.reload(importlib.import_module("Models.ann_utils"))
from Models.ann_utils import (
    create_dataset,
    compile_and_fit,
    compile_fit_evaluate,
    example_batch,
    create_dataset_lite,
    K_fold_training,
)

In this section we are going to introuce several Recurrent Neural Networks (RNNs). RNNs are a class of neural networks that are able to process sequences of inputs. They are widely used in Natural Language Processing (NLP) and in other fields where the input data is sequential, like in our case. After some basic experimentation non reported in the notebook, we experienced a great vanishing gradient problematic. For this reason we resolved to use GRU recurrent units with several architectures. Again the search strategy is not different from before but this time there are a few things to notice:
-   The `build_model` function varies from paragraph to paragraph

### 2.4.1 GRU - Raw Audio reduced by 1D-Conv - 10 classes

#### Create the dataset

In [ ]:
batch_size = 30

dataset, label = create_dataset_lite(df_ESC10, batch_size=batch_size, ndim=2)
INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns), verbose=1)

#### Build the model

In [ ]:
def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_filters=1,
    n_units=8,
    kernel_size=100,
    activation="tanh",
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):
    strides = int(kernel_size / 2)
    model = tf.keras.Sequential(
        [
            # reduce the dimensionality of the input with a 1DConvolution
            tf.keras.layers.Conv1D(
                filters=n_filters,
                kernel_size=kernel_size,
                strides=strides,
                activation=activation,
                input_shape=INPUT_DIM,
            ),
            # with more than 1 filters I'll have prpoblems with the channel dimension not accepted by Recurrent layers
            # apply a SimpleRNN layer
            tf.keras.layers.GRU(units=n_units, activation=activation),
            tf.keras.layers.Dense(n_labels, activation="softmax"),
        ]
    )
    if compile:
        # compile the model
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin" or in_colab
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    print(
        f"filters {n_filters}, kernel size {kernel_size}, units of GRU {n_units}, lr {learning_rate}, activation {activation}, strides {strides}"
    )
    print(f" Trainable parameters: {model.count_params()}")
    return model

#### Run a grid search to find the best params

In [ ]:
epochs = 50
patience = 10

# the grid search is too big to be implemented over all parameter space:

# Grid search for the 1DConv layer
print("FIRST GRID SEARCH")
params = {
    "n_filters": [1],
    "kernel_size": [64, 128, 256],
    "learning_rate": [1e-3, 1e-4],
}
# params = {'n_filters':[1],'kernel_size':[64], 'learning_rate':[1e-3, 1e-4]}
K_fold = 4
model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=1,
    K=K_fold,
)

In [ ]:
# best_params = {'kernel_size': 128, 'learning_rate': 0.001, 'n_filters': 1} #output of first grid search
# grid search for GRU layer
print("SECOND GRID SEARCH")
params = {
    "n_units": [8, 32, 128],
    "activation": ["tanh"],
    "learning_rate": [1e-3, 1e-2],
    "n_filters": [best_params["n_filters"]],
    "kernel_size": [best_params["kernel_size"]],
}
# params = {'n_units':[8],'activation':['tanh'], 'learning_rate':[1e-3, 1e-2], 'n_filters':[best_params['n_filters']], 'kernel_size':[best_params['kernel_size']]}

K_fold = 4
model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=1,
    K=K_fold,
)

In [ ]:
print(
    "The best params are:",
    {key: value for key, value in best_params.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_raw_241.pickle"), "wb"
) as handle:
    pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Train the model

In [ ]:
seed = 42
tf.random.set_seed(seed)

path_to_ESC10 = os.path.join(main_dir, "data", "ESC-10-depth")

# create the dataset
batch_size = 30
preprocessing = None
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC10,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=2,
)

In [ ]:
# We load the dictionary of the best parameters which are the MFCC ones with delta and delta_delta
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_raw_241.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

# refit only the best model
model = build_model(n_labels=n_labels, compile=False, **best_params)
learning_rate = best_params["learning_rate"]
epochs = 50
patience = 10
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=learning_rate)
)
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC10,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

### 2.4.2 GRU - STFT Audio preprocessed - 10 classes

#### Create the dataset

In [ ]:
batch_size = 30
preprocessing = "STFT"
dataset, label = create_dataset_lite(
    df_ESC10, batch_size=batch_size, preprocessing=preprocessing, ndim=2, transpose=True
)
INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns), verbose=1)

#### Build the model

In [ ]:
def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_units=8,
    activation="tanh",
    n_hidden_layers=1,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    model = tf.keras.Sequential()
    model.add(
        tf.keras.layers.GRU(
            units=n_units,
            activation=activation,
            input_shape=INPUT_DIM,
            return_sequences=True if n_hidden_layers > 1 else False,
        )
    )
    if n_hidden_layers > 0:
        for i in range(1, n_hidden_layers):
            model.add(
                tf.keras.layers.GRU(
                    units=n_units * (i + 1),
                    activation=activation,
                    input_shape=INPUT_DIM,
                    return_sequences=True if i < n_hidden_layers - 1 else False,
                )
            )
    model.add(tf.keras.layers.Dense(n_labels, activation="softmax"))
    if compile:
        # compile the model
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin"
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    print(
        f"units of GRU {n_units}, lr {learning_rate}, activation {activation}, n_hidden_layers {n_hidden_layers}"
    )

    return model

#### Run a grid search to find the best params

In [ ]:
epochs = 50
patience = 10
params = {
    "n_units": [8, 16, 32],
    "activation": ["tanh"],
    "learning_rate": [1e-3],
    "n_hidden_layers": [1, 2, 3],
}
# params = {'n_units':[8,16], 'activation':['tanh'], 'learning_rate':[1e-3], 'n_hidden_layers':[1]}

K_fold = 4

model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=0,
    K=K_fold,
)

In [ ]:
print(
    "The best params are:",
    {key: value for key, value in best_params.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_STFT_242.pickle"), "wb"
) as handle:
    pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Train the model

In [ ]:
# refit only the best model
# best_params = {'activation': 'tanh', 'learning_rate': 0.001, 'n_hidden_layers': 3, 'n_units': 16}
seed = 42
tf.random.set_seed(seed)

path_to_ESC10 = os.path.join(main_dir, "data", "ESC-10-depth")

# create the dataset
batch_size = 30
preprocessing = "STFT"
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC10,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=2,
)

In [ ]:
# We load the dictionary of the best parameters which are the MFCC ones with delta and delta_delta
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_STFT_242.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

# build the model with best parameters
model = build_model(n_labels=n_labels, compile=False, **best_params)
lr = best_params["learning_rate"]
epochs = 50
patience = 10
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC10,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=(
        tf.keras.optimizers.legacy.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    ),
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

### 2.4.3 GRU - MEL Audio preprocessed - 10 classes

#### Create the dataset

In [ ]:
batch_size = 30
preprocessing = "MEL"
dataset, label = create_dataset_lite(
    df_ESC10, batch_size=batch_size, preprocessing=preprocessing, ndim=2, transpose=True
)
INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns), verbose=0)

#### Build the model

In [ ]:
def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_units=8,
    activation="tanh",
    n_hidden_layers=1,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    model = tf.keras.Sequential()
    model.add(
        tf.keras.layers.GRU(
            units=n_units,
            activation=activation,
            input_shape=INPUT_DIM,
            return_sequences=True if n_hidden_layers > 1 else False,
        )
    )
    if n_hidden_layers > 0:
        for i in range(1, n_hidden_layers):
            model.add(
                tf.keras.layers.GRU(
                    units=n_units * (i + 1),
                    activation=activation,
                    input_shape=INPUT_DIM,
                    return_sequences=True if i < n_hidden_layers - 1 else False,
                )
            )
    model.add(tf.keras.layers.Dense(n_labels, activation="softmax"))

    if compile:
        # compile the model
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin"
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    print(
        f" units of GRU {n_units}, lr {learning_rate}, activation {activation}, n_hidden_layers {n_hidden_layers}"
    )

    return model

#### Run a grid search to find the best params

In [ ]:
epochs = 50
patience = 10
params = {
    "n_units": [8, 16, 32],
    "activation": ["tanh"],
    "learning_rate": [1e-3],
    "n_hidden_layers": [1, 2, 3],
}
# params = {'n_units':[8,12], 'activation':['tanh'], 'learning_rate':[1e-3], 'n_hidden_layers':[1]}

K_fold = 4

model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=0,
    K=K_fold,
)

In [ ]:
print(
    "The best params are:",
    {key: value for key, value in best_params.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_MEL_243.pickle"), "wb"
) as handle:
    pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Train the model

In [ ]:
# best_params = {'activation': 'tanh', 'learning_rate': 0.001, 'n_hidden_layers': 2, 'n_units': 32}

# refit only the best model
seed = 42
tf.random.set_seed(seed)
path_to_ESC10 = os.path.join(main_dir, "data", "ESC-10-depth")

# create the dataset
batch_size = 30
preprocessing = "MEL"
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC10,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=2,
)

In [ ]:
# We load the dictionary of the best parameters which are the MFCC ones with delta and delta_delta
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_MEL_243.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

# build the model with best parameters
model = build_model(n_labels=n_labels, compile=False, **best_params)
lr = best_params["learning_rate"]
epochs = 50
patience = 10
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=lr)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=lr)
)

model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC10,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=["accuracy"],  # ,'CategoricalAccuracy'],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

### 2.4.4 GRU - MFCC Audio preprocessed - 10 classes

#### Create the dataset

In [ ]:
batch_size = 30
preprocessing = "MFCC"
delta = False
delta_delta = False
dataset, label = create_dataset_lite(
    df_ESC10,
    batch_size=batch_size,
    preprocessing=preprocessing,
    delta=delta,
    delta_delta=delta_delta,
    ndim=2,
    transpose=True,
)
INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns), verbose=1)

#### Build the model

In [ ]:
def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_units=8,
    activation="tanh",
    order=2,  # order of the delta and delta_delta coefficients
    n_hidden_layers=1,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    model = tf.keras.Sequential()
    # eventually starts with a layer that computes the delta and delta_delta coefficients
    if order > 0:
        model.add(MFCCWithDeltaLayer(n_mfcc=40, order=order, input_shape=INPUT_DIM))
        model.add(
            tf.keras.layers.GRU(
                units=n_units,
                activation=activation,
                return_sequences=True if n_hidden_layers > 1 else False,
            )
        )
    elif order == 0:
        model.add(
            tf.keras.layers.GRU(
                units=n_units,
                activation=activation,
                input_shape=INPUT_DIM,
                return_sequences=True if n_hidden_layers > 1 else False,
            )
        )

    if n_hidden_layers > 0:
        for i in range(1, n_hidden_layers):
            model.add(
                tf.keras.layers.GRU(
                    units=n_units * (i + 1),
                    activation=activation,
                    input_shape=INPUT_DIM,
                    return_sequences=True if i < n_hidden_layers - 1 else False,
                )
            )
    model.add(tf.keras.layers.Dense(n_labels, activation="softmax"))

    if compile:
        # compile the model
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin" or in_colab
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    print(
        f" units of GRU {n_units}, lr {learning_rate}, activation {activation}, n_hidden_layers {n_hidden_layers}, order {order}"
    )

    return model

We split the grid search in two beacuse to use the GPU on colab the GRU cell must have tanh as activation function. Thus we tested ReLu locally and tanh on Google Colab Pro.

#### Run a grid search to find the best params


##### Activation ReLu

In [ ]:
epochs = 50
patience = 10
params = {
    "n_units": [8, 32],
    "activation": ["relu"],
    "learning_rate": [1e-3],
    "n_hidden_layers": [0, 1, 2],
    "order": [0, 1, 2],
}
# params = {'n_units':[8, 32], 'activation':['relu'], 'learning_rate':[1e-3], 'n_hidden_layers':[0],'order':[0]}

K_fold = 4

model_cv, result, best_params_relu = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=0,
    K=K_fold,
)

##### Activation Tanh

In [ ]:
epochs = 50
patience = 10
params = {
    "n_units": [8, 32],
    "activation": ["tanh"],
    "learning_rate": [1e-3],
    "n_hidden_layers": [0, 1, 2],
    "order": [0, 1, 2],
}
# params = {'n_units':[8, 32], 'activation':['tanh'], 'learning_rate':[1e-3], 'n_hidden_layers':[0],'order':[0]}

K_fold = 4

model_cv, result, best_params_tanh = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=0,
    K=K_fold,
)

#### Train the model

In [ ]:
print(
    "The best params are:",
    {key: value for key, value in best_params_tanh.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_MFCC_244.pickle"), "wb"
) as handle:
    pickle.dump(best_params_tanh, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# refit only the best model
seed = 42
tf.random.set_seed(seed)
path_to_ESC10 = os.path.join(main_dir, "data", "ESC-10-depth")

# create the dataset
batch_size = 30
preprocessing = "MFCC"
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC10,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    delta=False,
    delta_delta=False,
    verbose=0,
    show_example_batch=True,
    ndim=2,
)

In [ ]:
# We load the dictionary of the best parameters which are the MFCC ones with delta and delta_delta
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_MFCC_244.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

model = build_model(n_labels=n_labels, compile=False, **best_params)
epochs = 100
patience = 10
lr = best_params["learning_rate"]
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=lr)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=lr)
)
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC10,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=["accuracy"],  # ,'CategoricalAccuracy'],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

### 2.4.5 GRU with STFT audio preprocessed - 50 classes

#### Create the dataset

In [ ]:
seed = 42
tf.random.set_seed(seed)
path_to_ESC10 = os.path.join(main_dir, "data", "ESC-50-depth")

# best_params = {'activation': 'tanh', 'learning_rate': 0.001, 'n_hidden_layers': 3, 'n_units': 16}

# create the dataset
batch_size = 30
preprocessing = "STFT"  # the best preprocessing by grid search

# as always for the ESC50 it is better to save the dataset for the future calls
save_train_file = os.path.join(main_dir, "Saved_Datasets", "ESC50_STFT_train")
save_val_file = os.path.join(main_dir, "Saved_Datasets", "ESC50_STFT_val")
save_test_file = os.path.join(main_dir, "Saved_Datasets", "ESC50_STFT_test")
save_labels_file = os.path.join(main_dir, "Saved_Datasets", "ESC50_STFT_labels")
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC10,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=2,
    save_train=save_train_file,
    save_test=save_test_file,
    save_val=save_val_file,
    save_labels=save_labels_file,
)

We use the same build function used in 2.4.2 (with STFT prepocessing)

#### Build the model

In [ ]:
# We load the dictionary of the best parameters which are the MFCC ones with delta and delta_delta
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_STFT_242.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)


def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_units=8,
    activation="tanh",
    n_hidden_layers=1,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    model = tf.keras.Sequential()
    model.add(
        tf.keras.layers.GRU(
            units=n_units,
            activation=activation,
            input_shape=INPUT_DIM,
            return_sequences=True if n_hidden_layers > 1 else False,
        )
    )
    if n_hidden_layers > 0:
        for i in range(1, n_hidden_layers):
            model.add(
                tf.keras.layers.GRU(
                    units=n_units * (i + 1),
                    activation=activation,
                    input_shape=INPUT_DIM,
                    return_sequences=True if i < n_hidden_layers - 1 else False,
                )
            )
    model.add(tf.keras.layers.Dense(n_labels, activation="softmax"))
    if compile:
        # compile the model
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin"
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    print(
        f"units of GRU {n_units}, lr {learning_rate}, activation {activation}, n_hidden_layers {n_hidden_layers}"
    )

    return model


model = build_model(n_labels=n_labels, compile=False, **best_params)

#### Train the model

In [ ]:
# Train the model
epochs = 100
patience = 10
lr = best_params["learning_rate"]
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=lr)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=lr)
)
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

### 2.4.6 GRU with MFCC audio preprocessed - 50 classes

#### Create the dataset

In [ ]:
seed = 42
tf.random.set_seed(seed)
path_to_ESC10 = os.path.join(main_dir, "data", "ESC-50-depth")

# create the dataset
batch_size = 30
preprocessing = "MFCC"  # the best preprocessing by grid search
delta = False
delta_delta = False

# as always for the ESC50 it is better to save the dataset for the future calls
save_train_file = os.path.join(main_dir, "Saved_Datasets", "ESC50_MFCC_train")
save_val_file = os.path.join(main_dir, "Saved_Datasets", "ESC50_MFCC_val")
save_test_file = os.path.join(main_dir, "Saved_Datasets", "ESC50_MFCC_test")
save_labels_file = os.path.join(main_dir, "Saved_Datasets", "ESC50_MFCC_labels")
train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    path_to_ESC10,
    batch_size=batch_size,  # batch size
    preprocessing=preprocessing,
    verbose=0,
    show_example_batch=True,
    ndim=2,
    delta=delta,
    delta_delta=delta_delta,
    save_train=save_train_file,
    save_test=save_test_file,
    save_val=save_val_file,
    save_labels=save_labels_file,
)

We use the same build function used in 2.4.4 (with MFCC prepocessing)

#### Build the model

In [ ]:
# We load the dictionary of the best parameters which are the MFCC ones with delta and delta_delta
with open(
    os.path.join(main_dir, "Models", "best_params_RNN_MFCC_244.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)


def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_units=8,
    activation="tanh",
    order=2,  # order of the delta and delta_delta coefficients
    n_hidden_layers=1,
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):

    # Define Simple RNN  model
    model = tf.keras.Sequential()
    # eventually starts with a layer that computes the delta and delta_delta coefficients
    if order > 0:
        model.add(MFCCWithDeltaLayer(n_mfcc=40, order=order, input_shape=INPUT_DIM))
        model.add(
            tf.keras.layers.GRU(
                units=n_units,
                activation=activation,
                return_sequences=True if n_hidden_layers > 1 else False,
            )
        )
    elif order == 0:
        model.add(
            tf.keras.layers.GRU(
                units=n_units,
                activation=activation,
                input_shape=INPUT_DIM,
                return_sequences=True if n_hidden_layers > 1 else False,
            )
        )
    if n_hidden_layers > 0:
        for i in range(1, n_hidden_layers):
            model.add(
                tf.keras.layers.GRU(
                    units=n_units * (i + 1),
                    activation=activation,
                    input_shape=INPUT_DIM,
                    return_sequences=True if i < n_hidden_layers - 1 else False,
                )
            )
    model.add(tf.keras.layers.Dense(n_labels, activation="softmax"))

    if compile:
        # compile the model
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.legacy.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin" or in_colab
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    print(
        f" units of GRU {n_units}, lr {learning_rate}, activation {activation}, n_hidden_layers {n_hidden_layers}, order {order}"
    )

    return model


model = build_model(n_labels=n_labels, compile=False, **best_params)

#### Train the model

In [ ]:
lr = best_params["learning_rate"]
optimizer = (
    tf.keras.optimizers.legacy.Adam(learning_rate=lr)
    if sys.platform == "darwin" or in_colab
    else tf.keras.optimizers.Adam(learning_rate=lr)
)
# train the model
epochs = 100
patience = 10
model, history, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    metrics=["accuracy"],  # ,'CategoricalAccuracy'],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)